<a href="https://colab.research.google.com/github/parinyad123/financial-analyst-agent/blob/main/financial_analyst_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

financial_analyst_agent.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1cPuQuYhmfXjxkFifORwykG7_yhAR4yqt

# 📊 Financial Analyst Agent — Colab Dev Notebook

**Physics-informed financial analysis** ที่ผสม quantitative signals (Hurst exponent) กับ LLM reasoning ผ่าน ReAct agent

| Component | Technology |
|---|---|
| LLM (dev) | Groq — `openai/gpt-oss-120b` (ประหยัด Gemini quota) |
| Agent | LangGraph `create_react_agent` (ReAct pattern) |
| Market data | yfinance |
| Observability | LangSmith — project: `financial-analyst-agent` |

**Flow การทำงาน:**
```
User query → ReAct Agent (gpt-oss-120b)
                ├── get_stock_price      → yfinance
                ├── get_stock_financials → yfinance
                └── get_hurst_exponent   → yfinance + numpy (R/S analysis)
                        ↓
             LangSmith (trace ทุก step)
```

> ⚠️ **ก่อนรัน:** ตั้งค่า Colab Secrets (🔑 ไอคอนซ้ายมือ): `LANGCHAIN_API_KEY`, `GROQ_API_KEY` และเปิด Notebook access

**กฎการรัน:** รันจากบนลงล่างเท่านั้น — ถ้า Cell 2 (Verify) ไม่ผ่าน ห้ามรันต่อ

---
## Cell 1 — Install dependencies + Imports

ติดตั้งทุก package ในที่เดียว แล้ว import ทั้งหมด — รวมไว้ cell เดียวเพื่อไม่ให้เกิดปัญหา import order
(บทเรียนจากรอบ debug: langsmith cache ค่า env ตอน import ครั้งแรก ดังนั้นเราเลิกพึ่ง env แล้วใช้ explicit binding แทน → ดู Cell 2)

In [ ]:
# ============================================================
# Cell 1: Install + Imports
# ============================================================
!pip install -q -U langsmith langchain-groq langgraph yfinance langchain-core langchain-google-genai

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import time
from datetime import datetime

import numpy as np
import yfinance as yf
import json
import pandas as pd
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

from google.colab import userdata

# LangChain / LangGraph
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tracers import LangChainTracer
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent

# LangSmith
import langsmith
from langsmith import Client, traceable, tracing_context
from langsmith.run_helpers import get_current_run_tree

print("✅ All imports OK")
print("langsmith version:", langsmith.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 498.0/498.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.2/246.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00
✅ All imports OK
langsmith version: 0.8.15


---
## Cell 2 — LangSmith client (explicit binding)

**ทำไมไม่ใช้ env vars (`LANGCHAIN_TRACING_V2` ฯลฯ)?**
`langsmith.utils.get_env_var` ถูก cache ด้วย `lru_cache` — ถ้า import ก่อน set env ค่าจะค้างเป็น "disabled" ตลอด session ใน notebook ที่ cell order ไม่แน่นอน นี่คือระเบิดเวลา

**แนวทางที่ใช้:** bind `api_key` + `project_name` ตรง ๆ เข้า `Client`, `LangChainTracer`, และ `@traceable` ทุกจุด — ไม่พึ่ง env เลย
(ตอน deploy จริงใน Docker/FastAPI ค่อยกลับไปใช้ env ได้ เพราะ env ถูก set ตอน container start ก่อน import เสมอ)

**Assertion ท้าย cell:** ถ้า print ❌ ห้ามรัน cell ถัดไป — เช็ค API key ใน Colab Secrets ก่อน

In [ ]:
# ============================================================
# Cell 2: LangSmith client + tracer (explicit binding — ไม่พึ่ง env)
# ============================================================
PROJECT_NAME = "financial-analyst-agent"

ls_client = Client(
    api_key=userdata.get("LANGCHAIN_API_KEY"),
    api_url="https://api.smith.langchain.com",
)

# tracer สำหรับ LangGraph agent — ผูก project + client ตรง ๆ
tracer = LangChainTracer(
    project_name=PROJECT_NAME,
    client=ls_client,
)

# สร้าง project ถ้ายังไม่มี + ใช้เป็น connectivity check ไปในตัว
try:
    project = ls_client.read_project(project_name=PROJECT_NAME)
    print(f"✅ Project exists: {project.name}")
except Exception:
    project = ls_client.create_project(
        PROJECT_NAME,
        description="Financial Analyst Agent with ReAct + Hurst Exponent",
    )
    print(f"✅ Project created: {project.name}")

# 🛑 Gate: ต้องผ่านก่อนรันต่อ
assert project is not None, "❌ เชื่อม LangSmith ไม่ได้ — เช็ค LANGCHAIN_API_KEY ใน Colab Secrets"
print("✅ LangSmith ready — ไปต่อได้")

✅ Project exists: financial-analyst-agent
✅ LangSmith ready — ไปต่อได้


---
## Cell 3 — Tools: price, financials, Hurst exponent

**Pattern สำคัญ — `@tool` นอกสุด, `@traceable` ห่อ logic ข้างใน:**

```python
@tool                      # ← LLM เห็น docstring นี้ ใช้ตัดสินใจเลือก tool
def tool_name(x):
    return _logic(x)

@traceable(run_type="tool", client=ls_client)   # ← LangSmith trace ตัวนี้
def _logic(x): ...
```

แยกกันเพราะ: `@tool` ทำหน้าที่ interface กับ LLM (schema + docstring) ส่วน `@traceable` ทำหน้าที่ observability — ถ้อยซ้อน decorator บนฟังก์ชันเดียวกันจะตีกัน

| Tool | ข้อมูลที่คืน | Tags ใน LangSmith |
|---|---|---|
| `get_stock_price` | ราคา, % change, 52W range, P/E, market cap | `market-data`, `yfinance` |
| `get_stock_financials` | revenue, net income, margin, growth, EPS, D/E | `fundamentals`, `yfinance` |
| `get_hurst_exponent` | Hurst exponent (R/S analysis) + market regime | `quant`, `regime-detection` |

**Hurst exponent อ่านยังไง:** H > 0.55 → Trending (momentum ใช้ได้) | H < 0.45 → Mean-Reverting (RSI/Bollinger) | ระหว่างนั้น → Random Walk

In [ ]:
# ============================================================
# Cell 3: Tools — get_stock_price / get_stock_financials / get_hurst_exponent
# ============================================================

# ---------- Tool 1: ราคาปัจจุบัน + key metrics ----------
@tool
def get_stock_price(ticker: str) -> str:
    """Fetch CURRENT PRICE and trading metrics: price, % change, 52W range,
    market cap, position in 52W range.
    Use ONLY when the user asks about price, % change, or where the stock
    trades in its range. Do NOT call this for fundamentals like revenue,
    net income, profit margin, EPS, or debt — use get_stock_financials for those."""
    return _fetch_stock_price_logic(ticker)

@traceable(
    name="fetch_stock_price",
    run_type="tool",
    tags=["market-data", "yfinance"],
    client=ls_client,
)
def _fetch_stock_price_logic(ticker: str) -> str:
    try:
        stock = yf.Ticker(ticker.upper())
        hist = stock.history(period="5d")   # 5d เผื่อวันที่ market ปิด
        if hist.empty:
            return f"No data for {ticker}"

        latest = hist['Close'].iloc[-1]
        prev = hist['Close'].iloc[-2] if len(hist) > 1 else latest
        change_pct = ((latest - prev) / prev) * 100
        info = stock.info

        pos_in_range = (latest - info.get('fiftyTwoWeekLow', latest)) / \
               (info.get('fiftyTwoWeekHigh', latest) - info.get('fiftyTwoWeekLow', latest) + 1e-9) * 100

        return (
            f"Ticker: {ticker.upper()}\n"
            f"Price: ${latest:.2f} (Change: {change_pct:+.2f}%)\n"
            f"52W Range: ${info.get('fiftyTwoWeekLow','N/A')} – ${info.get('fiftyTwoWeekHigh','N/A')}\n"
            f"P/E (TTM): {info.get('trailingPE','N/A')} | Forward P/E: {info.get('forwardPE','N/A')}\n"
            f"Market Cap: ${info.get('marketCap',0)/1e9:.1f}B\n"
            f"Position in 52W range: {pos_in_range:.0f}%"
        )
    except Exception as e:
        return f"Error: {str(e)}"


# ---------- Tool 2: Fundamentals ----------
@tool
def get_stock_financials(ticker: str) -> str:
    """Get fundamental financial metrics for analysis."""
    return _fetch_financials_logic(ticker)

@traceable(
    name="fetch_financials",
    run_type="tool",
    tags=["fundamentals", "yfinance"],
    client=ls_client,
)
def _fetch_financials_logic(ticker: str) -> str:
    try:
        info = yf.Ticker(ticker.upper()).info
        return (
            f"Revenue (TTM): ${info.get('totalRevenue',0)/1e9:.1f}B\n"
            f"Net Income: ${info.get('netIncomeToCommon',0)/1e9:.1f}B\n"
            f"Profit Margin: {info.get('profitMargins',0)*100:.1f}%\n"
            f"Revenue Growth YoY: {info.get('revenueGrowth',0)*100:.1f}%\n"
            f"EPS (TTM): ${info.get('trailingEps','N/A')}\n"
            f"Debt/Equity: {info.get('debtToEquity','N/A')}"
        )
    except Exception as e:
        return f"Error: {str(e)}"


# ---------- Tool 3: Hurst exponent (R/S analysis) ----------
@tool
def get_hurst_exponent(ticker: str) -> str:
    """Calculate Hurst exponent to detect market regime."""
    return _calc_hurst_logic(ticker)

@traceable(
    name="calc_hurst_exponent",
    run_type="tool",
    tags=["quant", "regime-detection"],
    client=ls_client,
)
def _calc_hurst_logic(ticker: str) -> str:
    try:
        # 1) log returns จาก 1Y daily close
        hist = yf.Ticker(ticker.upper()).history(period="1y")["Close"]
        returns = np.log(hist / hist.shift(1)).dropna().values

        # 2) Rescaled Range (R/S) ต่อ lag — แบ่ง series เป็น segments
        lags = range(2, 20)
        rs_values = []
        for lag in lags:
            segments = [returns[i:i+lag] for i in range(0, len(returns)-lag, lag)]
            rs_list = [
                (np.max(np.cumsum(s - np.mean(s))) - np.min(np.cumsum(s - np.mean(s)))) / np.std(s)
                for s in segments if np.std(s) > 0
            ]
            if rs_list:
                rs_values.append(np.mean(rs_list))

        # 3) Hurst = slope ของ log(R/S) vs log(lag)
        hurst = np.polyfit(np.log(list(lags)[:len(rs_values)]), np.log(rs_values), 1)[0]

        # 4) จำแนก regime
        if hurst > 0.55:
            regime = "📈 Trending — momentum strategies work"
        elif hurst < 0.45:
            regime = "↔️ Mean-Reverting — RSI/Bollinger strategies work"
        else:
            regime = "🎲 Random Walk — harder to predict"

        return f"Hurst Exponent ({ticker.upper()}, 1Y): {hurst:.4f}\nRegime: {regime}"
    except Exception as e:
        return f"Error: {str(e)}"


print("✅ Tools ready:", [t.name for t in [get_stock_price, get_stock_financials, get_hurst_exponent]])

✅ Tools ready: ['get_stock_price', 'get_stock_financials', 'get_hurst_exponent']


---
## Cell 4–6 — Tools เพิ่มเติม (ตาม build order)

- **Cell 4:** `analyze_portfolio_risk` (UC-2a) — Volatility, Sharpe, Sortino, VaR/CVaR 95%, Max Drawdown, correlation matrix
- **Cell 5:** `search_market_news` — Gemini 2.5-flash + Google Search grounding (ใช้ model แยกจาก agent หลัก)
- **Cell 6:** `track_portfolio` (UC-2b) — dev ด้วย `MOCK_PORTFOLIO` dict ก่อน, swap เป็น PostgreSQL ตอน deploy

In [ ]:
# ============================================================
# Cell 4: Tool — analyze_portfolio_risk (UC-2a)
# Input: {ticker: weight}, weights ต้องรวม 1.0 ± 0.01
# ============================================================

TRADING_DAYS = 252
RISK_FREE_RATE = 0.045      # ~3M T-Bill — v1 hardcode, v2 ค่อยดึง ^IRX จาก yfinance
MIN_HISTORY_DAYS = 60       # ขั้นต่ำที่ metrics พอเชื่อถือได้ (กัน ticker เพิ่ง IPO)


@tool
def analyze_portfolio_risk(portfolio: str) -> str:
    """USE THIS TOOL for any portfolio risk assessment question
    (e.g. "ประเมิน risk ของ portfolio", "วิเคราะห์ความเสี่ยงพอร์ต").
    Computes ALL risk metrics in one call: volatility, Sharpe, Sortino,
    VaR/CVaR 95%, max drawdown, correlation matrix.
    Do NOT call get_stock_price or get_stock_financials for portfolio risk —
    this tool fetches all required data itself.
    Input MUST be a JSON string: '{"NVDA": 0.5, "AMD": 0.3, "TSLA": 0.2}'.
    Weights must sum to 1.0."""
    return _portfolio_risk_logic(portfolio)


@traceable(
    name="portfolio_risk_analysis",
    run_type="tool",
    tags=["quant", "risk"],
    client=ls_client,
)
def _portfolio_risk_logic(portfolio) -> str:
    # ---------- 1) Parse + validate input ----------
    # เผื่อ model ส่ง dict มาตรง ๆ (บาง model ไม่ serialize เป็น string)
    if isinstance(portfolio, dict):
        weights_dict = portfolio
    else:
        try:
            weights_dict = json.loads(portfolio)
        except (json.JSONDecodeError, TypeError) as e:
            return f'Error: invalid input — ต้องเป็น JSON string เช่น {{"NVDA": 0.5, "AMD": 0.5}} ({e})'

    if not isinstance(weights_dict, dict) or not weights_dict:
        return "Error: portfolio ว่าง หรือ format ไม่ใช่ {ticker: weight}"

    try:
        weights_dict = {str(t).upper().strip(): float(w) for t, w in weights_dict.items()}
    except (ValueError, TypeError):
        return "Error: weight ทุกตัวต้องเป็นตัวเลข เช่น 0.5 ไม่ใช่ '50%'"

    if any(w <= 0 for w in weights_dict.values()):
        return "Error: weight ต้องเป็นบวกทุกตัว (v1 ยังไม่รองรับ short positions)"

    total = sum(weights_dict.values())
    if abs(total - 1.0) > 0.01:
        return (
            f"Error: weights รวมได้ {total:.3f} ต้องรวม 1.0 ± 0.01 — "
            f"ให้แจ้ง user ปรับ weights เอง (tool จะไม่ normalize ให้)"
        )

    tickers = sorted(weights_dict)   # yf.download คืน columns เรียง alphabet — sort ให้ตรงกันไว้ก่อน

    # ---------- 2) Download 1Y daily close ----------
    try:
        data = yf.download(tickers, period="1y", progress=False, auto_adjust=True)["Close"]
    except Exception as e:
        return f"Error: ดึงข้อมูลจาก yfinance ไม่สำเร็จ — {e}"

    # Edge: ticker เดียว → yfinance อาจคืน Series ไม่ใช่ DataFrame
    if isinstance(data, pd.Series):
        data = data.to_frame(name=tickers[0])

    # Edge: ticker พิมพ์ผิด / delisted → คอลัมน์เป็น NaN ทั้งแถบ
    # ชั้นแรก: retry เฉพาะตัวที่หาย (ใส่ใน _portfolio_risk_logic และ _track_portfolio_logic)
    dead = [t for t in tickers if t not in data.columns or data[t].isna().all()]
    if dead:
        time.sleep(1.5)
        retry = yf.download(dead, period="1y", progress=False, auto_adjust=True)["Close"]
        if isinstance(retry, pd.Series):
            retry = retry.to_frame(name=dead[0])
        for t in dead:
            if t in retry.columns and not retry[t].isna().all():
                data[t] = retry[t]
        dead = [t for t in tickers if t not in data.columns or data[t].isna().all()]

    # ชั้นสอง: error message บอกความจริงทั้งสองทาง
    if dead:
        return (f"Error: ไม่พบข้อมูลของ {dead} หลัง retry — "
                f"อาจเป็น ticker ผิด/delisted หรือ API ขัดข้องชั่วคราว "
                f"(ถ้ามั่นใจว่า ticker ถูก ให้ลองใหม่อีกครั้ง)")

    # Align dates: ตัดวันที่บาง ticker ไม่มีข้อมูล (IPO ใหม่ / ตลาดคนละประเทศ)
    data = data[tickers].dropna()
    if len(data) < MIN_HISTORY_DAYS:
        return (
            f"Error: ข้อมูลที่ทุก ticker มีร่วมกันมีแค่ {len(data)} วัน "
            f"(ขั้นต่ำ {MIN_HISTORY_DAYS}) — อาจมี ticker ที่เพิ่ง IPO"
        )

    # ---------- 3) Returns + portfolio series ----------
    returns = np.log(data / data.shift(1)).dropna()

    # ⚠️ จุดพังคลาสสิก: weights ต้อง align ตาม column order ของ DataFrame
    # ไม่ใช่ order ใน dict ที่ user ส่งมา
    weights = np.array([weights_dict[t] for t in data.columns])
    port_returns = returns.values @ weights

    # ---------- 4) Metrics (annualized, 252 trading days) ----------
    ann_return = port_returns.mean() * TRADING_DAYS
    ann_vol = port_returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe = (ann_return - RISK_FREE_RATE) / ann_vol if ann_vol > 0 else float("nan")

    # Edge: portfolio ที่แทบไม่มีวันติดลบ → downside std = 0 → division by zero
    downside = port_returns[port_returns < 0]
    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino_str = f"{(ann_return - RISK_FREE_RATE) / (downside.std(ddof=1) * np.sqrt(TRADING_DAYS)):.2f}"
    else:
        sortino_str = "N/A (แทบไม่มีวันติดลบใน 1Y)"

    var_95 = np.percentile(port_returns, 5)
    cvar_95 = port_returns[port_returns <= var_95].mean()

    # Max Drawdown — port_returns เป็น log returns → ใช้ exp(cumsum) ตรงกว่า cumprod(1+r)
    cum = np.exp(np.cumsum(port_returns))
    peak = np.maximum.accumulate(cum)
    max_dd = ((cum - peak) / peak).min()

    # Per-ticker annualized volatility
    per_vol = returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
    per_vol_str = "\n".join(f"  {t}: {v * 100:.1f}%" for t, v in per_vol.items())

    # Edge: correlation matrix ต้องมี >= 2 tickers
    if len(tickers) >= 2:
        corr_str = returns.corr().round(2).rename_axis(index=None, columns=None).to_string()
    else:
        corr_str = "N/A (single asset — ไม่มี correlation)"

    return (
        f"Portfolio: {weights_dict}\n"
        f"Period: 1Y daily ({len(returns)} trading days)\n"
        f"Annualized Return: {ann_return * 100:+.1f}%\n"
        f"Annualized Volatility: {ann_vol * 100:.1f}%\n"
        f"Sharpe Ratio: {sharpe:.2f} (rf = {RISK_FREE_RATE * 100:.1f}%)\n"
        f"Sortino Ratio: {sortino_str}\n"
        f"VaR 95% (daily): {var_95 * 100:.2f}% | CVaR 95% (daily): {cvar_95 * 100:.2f}%\n"
        f"Max Drawdown: {max_dd * 100:.1f}%\n"
        f"Per-Ticker Volatility (annualized):\n{per_vol_str}\n"
        f"Correlation Matrix:\n{corr_str}"
    )


print("✅ analyze_portfolio_risk ready")

# ---------- Smoke test: เรียก logic ตรง ๆ ก่อนผ่าน agent ----------
# (เทสนอก agent ก่อน — แยกปัญหา "logic พัง" ออกจาก "model เรียก tool เพี้ยน")
print("\n--- Test 1: happy path ---")
print(_portfolio_risk_logic('{"NVDA": 0.5, "AMD": 0.3, "TSLA": 0.2}'))

print("\n--- Test 2: weights ไม่ครบ 1.0 ---")
print(_portfolio_risk_logic('{"NVDA": 0.5, "AMD": 0.3}'))

print("\n--- Test 3: ticker มั่ว ---")
print(_portfolio_risk_logic('{"ZZZFAKE123": 1.0}'))

print("\n--- Test 4: single ticker ---")
print(_portfolio_risk_logic('{"NVDA": 1.0}'))

print("\n--- Test 5: JSON พัง ---")
print(_portfolio_risk_logic("NVDA 50% AMD 50%"))



✅ analyze_portfolio_risk ready

--- Test 1: happy path ---
Portfolio: {'NVDA': 0.5, 'AMD': 0.3, 'TSLA': 0.2}
Period: 1Y daily (250 trading days)
Annualized Return: +68.0%
Annualized Volatility: 36.7%
Sharpe Ratio: 1.73 (rf = 4.5%)
Sortino Ratio: 2.44
VaR 95% (daily): -3.84% | CVaR 95% (daily): -5.17%
Max Drawdown: -22.5%
Per-Ticker Volatility (annualized):
  AMD: 65.6%
  NVDA: 35.0%
  TSLA: 44.6%
Correlation Matrix:
       AMD  NVDA  TSLA
AMD   1.00  0.49  0.36
NVDA  0.49  1.00  0.37
TSLA  0.36  0.37  1.00

--- Test 2: weights ไม่ครบ 1.0 ---
Error: weights รวมได้ 0.800 ต้องรวม 1.0 ± 0.01 — ให้แจ้ง user ปรับ weights เอง (tool จะไม่ normalize ให้)

--- Test 3: ticker มั่ว ---
Error: ไม่พบข้อมูลของ ['ZZZFAKE123'] หลัง retry — อาจเป็น ticker ผิด/delisted หรือ API ขัดข้องชั่วคราว (ถ้ามั่นใจว่า ticker ถูก ให้ลองใหม่อีกครั้ง)

--- Test 4: single ticker ---
Portfolio: {'NVDA': 1.0}
Period: 1Y daily (250 trading days)
Annualized Return: +37.3%
Annualized Volatility: 35.0%
Sharpe Ratio: 0.94 (rf

In [ ]:
# Gate check ก่อนเริ่ม Cell 5 — ใน Colab
from langchain_google_genai import ChatGoogleGenerativeAI
m = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=userdata.get("GOOGLE_API_KEY"))
print(m.invoke("ping").content)   # ผ่าน = มี quota จริง, 429 = เลื่อน search เป็น v2

Pong!


In [ ]:
# ============================================================
# Cell 5: Tool — search_market_news (Gemini + Google Search grounding)
# ============================================================
# ⚠️ model นี้ "แยก" จาก agent หลัก (Groq) — agent เรียก tool นี้
#    แล้ว tool ยิง Gemini ข้างในเอง. Gemini ล่ม/quota หมด = คืน error
#    string graceful (ไม่ทำ agent graph พัง) ตาม pattern เดียวกับ tool อื่น
from langchain_google_genai import ChatGoogleGenerativeAI

# สร้าง model ครั้งเดียวระดับ module — .bind() ผูก google_search tool ติดไป
# (langchain-google-genai 4.x: dict ตรง ๆ ไม่ต้อง import GenAITool แล้ว)
_news_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",          # validate แล้วใน gate check; 2.0-flash ตายไปแล้ว 1 มิ.ย. 2026
    temperature=0.1,
    api_key=userdata.get("GOOGLE_API_KEY"),
).bind(tools=[{"google_search": {}}])  # ⚠️ ห้ามใส่ response_schema คู่กัน → grounding_chunks จะว่าง


def _extract_grounding_sources(resp, max_sources=5):
    """ดึงแหล่งข่าวจาก grounding_metadata — เก็บแค่ domain (title)
    ไม่เอา redirect URL เพราะ:
      1) URL เป็น vertexaisearch redirect ที่หมดอายุใน ~ไม่กี่วัน (ไม่มี publisher URL ตรง)
      2) resolve redirect = ยิง HTTP เพิ่ม + เข้าพื้นที่สีเทาของ grounding ToS
    👉 รันครั้งแรก: print(resp.response_metadata) เพื่อ confirm path"""
    meta = getattr(resp, "response_metadata", {}) or {}
    gm = meta.get("grounding_metadata") or {}
    chunks = gm.get("grounding_chunks") or []

    seen, out = set(), []
    for c in chunks:
        web = c.get("web") if isinstance(c, dict) else getattr(c, "web", None)
        if web is None:
            continue
        title = web.get("title") if isinstance(web, dict) else getattr(web, "title", None)
        if title:
            title = title.strip()
            if title and title not in seen:
                seen.add(title)
                out.append(title)
        if len(out) >= max_sources:
            break
    return out


@tool
def search_market_news(query: str) -> str:
    """USE THIS TOOL when the question needs CURRENT EVENTS, news, analyst
    commentary, or qualitative context that is NOT a number
    (e.g. "มีข่าวอะไรเกี่ยวกับ NVDA", "ทำไมหุ้นร่วง", "analyst มองยังไง",
    earnings reactions, M&A, regulatory news).
    Do NOT call this for numeric data — use get_stock_price / get_stock_financials
    for price, P/E, revenue, margins. Do NOT call for risk metrics —
    use analyze_portfolio_risk. Input: a natural-language search query string."""
    return _search_news_logic(query)


@traceable(
    name="search_market_news",
    run_type="tool",
    tags=["search", "news", "gemini"],
    client=ls_client,
)
def _search_news_logic(query: str) -> str:
    try:
        today = datetime.now().strftime("%Y-%m-%d")
        prompt = (
            f"Today is {today}. Search for the most recent news, analyst commentary, "
            f"and market-moving events about: {query}\n"
            f"Summarize in 4-6 concise bullet points, prioritizing items from the "
            f"last 2 weeks and including dates. If nothing recent is found, say so "
            f"explicitly instead of returning old/generic info. "
            f"Reply in Thai mixed with English financial terms. "
            f"Do not give price targets or buy/sell recommendations."
        )
        resp = _news_model.invoke(prompt)

        # gemini อาจคืน content เป็น str หรือ list-of-parts → normalize
        content = resp.content
        if isinstance(content, list):
            content = "".join(
                p.get("text", "") if isinstance(p, dict) else str(p) for p in content
            )
        content = (content or "").strip()

        sources = _extract_grounding_sources(resp)
        if sources:
            content += "\n\nSources:\n" + "\n".join(f"  - {s}" for s in sources)

        return content or "No recent news found."

    except Exception as e:
        # quota 429 / ResourceExhausted / network → graceful (agent อ่าน string นี้
        # แล้วบอก user ว่าดึงข่าวไม่ได้ ไม่ไปเดาข่าวเองตาม system prompt rule)
        return f"Error: ดึงข่าวไม่สำเร็จ — {type(e).__name__}: {e}"


print("✅ search_market_news ready")

# ---------- Smoke tests: logic ตรง ๆ ก่อนผ่าน agent ----------
print("\n--- Test 1: ข่าวหุ้นตัวเดียว ---")
print(_search_news_logic("NVDA latest news and analyst reactions"))

print("\n--- Test 2: sector / macro theme ---")
print(_search_news_logic("US semiconductor export restrictions to China"))

print("\n--- Test 3: query ที่ไม่ควรมีข่าว (เช็ค 'say so explicitly') ---")
print(_search_news_logic("ZZZFAKE123 stock news"))

✅ search_market_news ready

--- Test 1: ข่าวหุ้นตัวเดียว ---
นี่คือสรุปข่าวสารล่าสุด, บทวิเคราะห์จากนักวิเคราะห์ และเหตุการณ์สำคัญที่ส่งผลต่อตลาดเกี่ยวกับ NVDA โดยเน้นข้อมูลในช่วง 2 สัปดาห์ที่ผ่านมา:

*   **Analyst Sentiment และ Credit Rating Upgrade:** นักวิเคราะห์ส่วนใหญ่ยังคงมีมุมมองเชิงบวก (bullish) ต่อ NVIDIA โดยบางรายมองว่าหุ้นยังคงมีมูลค่าต่ำกว่าที่ควรจะเป็น (undervalued) แม้จะมีการเติบโตที่แข็งแกร่ง นอกจากนี้ S&P Global Ratings ได้ปรับเพิ่มอันดับเครดิตของ NVDA เป็น 'AA' เมื่อวันที่ 11 มิถุนายน 2026 โดยอ้างถึงการเติบโตที่ขับเคลื่อนด้วย AI ที่แข็งแกร่งและกระแสเงินสดจำนวนมาก
*   **Strategic Partnerships และ Product Expansion:** NVIDIA ยังคงขยาย AI ecosystem ผ่านการเป็นพันธมิตรใหม่และการเปิดตัวผลิตภัณฑ์ โดยมีรายงานเมื่อวันที่ 13 มิถุนายน 2026 ว่า NVIDIA เริ่มเสนอขาย Vera CPUs ให้กับลูกค้าชาวจีน และมีการประกาศความร่วมมือด้านเทคโนโลยีหลายปีกับ SK hynix เพื่อพัฒนาหน่วยความจำสำหรับ AI factories เมื่อวันที่ 7 มิถุนายน 2026 รวมถึงความร่วมมือเชิงกลยุทธ์ 6 ปีกับ Sharon AI เพื่อสร้าง AI fac

In [ ]:
# ============================================================
# Cell 6: Tool — track_portfolio (UC-2b)
# ============================================================
# Dev: MOCK_PORTFOLIOS dict (โครงสร้าง mirror ตาราง positions ใน SQLite)
# ตอนทำ FastAPI: เปลี่ยนแค่ _load_positions() ให้ query DB —
# tool interface + logic ที่เหลือไม่ขยับ (นี่คือเหตุผลที่แยก seam ไว้)

# keyed ด้วย portfolio_id เพื่อ mimic DB lookup ตั้งแต่ตอน mock
MOCK_PORTFOLIOS = {
    "demo": {
        "name": "Demo Tech Portfolio",
        "positions": [
            {"ticker": "NVDA", "shares": 10, "avg_cost": 150.0},
            {"ticker": "AMD",  "shares": 20, "avg_cost": 100.0},
            {"ticker": "TSLA", "shares":  5, "avg_cost": 200.0},
        ],
    },
    # สำหรับเทส edge case: ticker ที่ fetch ราคาไม่ได้
    "test_dead_ticker": {
        "name": "Test — delisted ticker",
        "positions": [
            {"ticker": "NVDA",       "shares": 10, "avg_cost": 150.0},
            {"ticker": "ZZZFAKE123", "shares":  5, "avg_cost": 50.0},
        ],
    },
}


def _load_positions(portfolio_id: str):
    """Data-access seam — v1: dict lookup | FastAPI: SELECT จาก SQLite
    คืน (portfolio_name, positions) หรือ raise KeyError ถ้าไม่เจอ"""
    pf = MOCK_PORTFOLIOS[portfolio_id]
    return pf["name"], pf["positions"]


@tool
def track_portfolio(portfolio_id: str) -> str:
    """USE THIS TOOL for tracking an EXISTING portfolio's performance
    (e.g. "พอร์ตของฉันกำไรเท่าไหร่", "track portfolio", "P&L ตอนนี้",
    "ขาดทุนอยู่เท่าไหร่"). Loads saved positions by portfolio_id,
    fetches current prices, and computes unrealized P&L per position,
    total market value, total P&L, and current weights.
    Do NOT call get_stock_price separately — this tool fetches all prices itself.
    Input: portfolio_id string, e.g. "demo".
    Note: this is different from analyze_portfolio_risk, which assesses
    risk of a HYPOTHETICAL weight allocation before investing."""
    return _track_portfolio_logic(portfolio_id)


@traceable(
    name="track_portfolio",
    run_type="tool",
    tags=["portfolio", "tracking"],
    client=ls_client,
)
def _track_portfolio_logic(portfolio_id: str) -> str:
    # ---------- 1) Load positions ----------
    try:
        portfolio_id = str(portfolio_id).strip()
        pf_name, positions = _load_positions(portfolio_id)
    except KeyError:
        available = ", ".join(MOCK_PORTFOLIOS.keys())
        return (
            f"Error: ไม่พบ portfolio_id '{portfolio_id}' — "
            f"portfolio ที่มีอยู่: {available}. ให้ถาม user ว่าหมายถึงอันไหน"
        )

    if not positions:
        return f"Portfolio '{pf_name}' ({portfolio_id}) ยังไม่มี position ใด ๆ"

    # Validate ข้อมูลใน "DB" — กันข้อมูลเสียหลุดเข้ามาคำนวณ
    for p in positions:
        if p.get("shares", 0) <= 0 or p.get("avg_cost", 0) <= 0:
            return (
                f"Error: position {p.get('ticker', '?')} มี shares/avg_cost "
                f"ไม่ถูกต้อง ({p}) — ข้อมูลใน DB อาจเสียหาย"
            )

    # ---------- 2) Batch fetch ราคาปัจจุบัน (ครั้งเดียวทุก ticker) ----------
    tickers = sorted({p["ticker"].upper() for p in positions})  # unique — เผื่อมีหลาย lot ของหุ้นเดียวกัน
    try:
        data = yf.download(tickers, period="5d", progress=False, auto_adjust=True)["Close"]
    except Exception as e:
        return f"Error: ดึงราคาจาก yfinance ไม่สำเร็จ — {e}"

    if isinstance(data, pd.Series):          # ticker เดียว → Series
        data = data.to_frame(name=tickers[0])

    # ราคาล่าสุดต่อ ticker — NaN = fetch ไม่ได้ (delisted / ticker ผิด)
    prices = {}
    for t in tickers:
        if t in data.columns and data[t].dropna().size > 0:
            prices[t] = float(data[t].dropna().iloc[-1])
        else:
            prices[t] = None                 # อย่า fail ทั้งพอร์ตเพราะตัวเดียว

    # ---------- 3) P&L per position ----------
    lines, warnings_list = [], []
    total_mv = total_cost = 0.0

    for p in positions:
        t = p["ticker"].upper()
        shares, avg_cost = p["shares"], p["avg_cost"]
        cost_basis = shares * avg_cost

        if prices[t] is None:
            warnings_list.append(
                f"  ⚠️ {t}: price unavailable (delisted/ticker ผิด?) — "
                f"{shares} shares @ cost ${avg_cost:.2f} ไม่รวมในยอด"
            )
            continue

        mv = shares * prices[t]
        pnl = mv - cost_basis
        pnl_pct = pnl / cost_basis * 100
        total_mv += mv
        total_cost += cost_basis

        lines.append(
            f"  {t}: {shares:g} shares @ avg ${avg_cost:.2f} → "
            f"now ${prices[t]:.2f} | MV ${mv:,.2f} | "
            f"P&L {pnl:+,.2f} ({pnl_pct:+.1f}%)"
        )

    if total_mv == 0:
        return (
            f"Portfolio '{pf_name}': ดึงราคาไม่ได้สักตัวเดียว\n"
            + "\n".join(warnings_list)
        )

    # ---------- 4) Current weights (จาก market value จริงตอนนี้) ----------
    weight_lines = []
    for p in positions:
        t = p["ticker"].upper()
        if prices[t] is not None:
            w = (p["shares"] * prices[t]) / total_mv * 100
            weight_lines.append(f"  {t}: {w:.1f}%")

    total_pnl = total_mv - total_cost
    total_pnl_pct = total_pnl / total_cost * 100

    out = (
        f"Portfolio: {pf_name} (id: {portfolio_id})\n"
        f"Positions:\n" + "\n".join(lines) + "\n"
        f"Total Market Value: ${total_mv:,.2f}\n"
        f"Total Cost Basis: ${total_cost:,.2f}\n"
        f"Total Unrealized P&L: {total_pnl:+,.2f} ({total_pnl_pct:+.1f}%)\n"
        f"Current Weights (by market value):\n" + "\n".join(weight_lines)
    )
    if warnings_list:
        out += "\nWarnings:\n" + "\n".join(warnings_list)
    return out


print("✅ track_portfolio ready")

# ---------- Smoke tests: logic ตรง ๆ ก่อนผ่าน agent ----------
print("\n--- Test 1: happy path (demo) ---")
print(_track_portfolio_logic("demo"))

print("\n--- Test 2: portfolio_id ไม่มีอยู่ ---")
print(_track_portfolio_logic("nonexistent_id"))

print("\n--- Test 3: มี ticker ที่ fetch ราคาไม่ได้ ---")
print(_track_portfolio_logic("test_dead_ticker"))

print("\n--- Test 4: id มี whitespace (agent อาจส่งมาแบบนี้) ---")
print(_track_portfolio_logic("  demo  "))

✅ track_portfolio ready

--- Test 1: happy path (demo) ---
Portfolio: Demo Tech Portfolio (id: demo)
Positions:
  NVDA: 10 shares @ avg $150.00 → now $205.19 | MV $2,051.90 | P&L +551.90 (+36.8%)
  AMD: 20 shares @ avg $100.00 → now $511.57 | MV $10,231.40 | P&L +8,231.40 (+411.6%)
  TSLA: 5 shares @ avg $200.00 → now $406.43 | MV $2,032.15 | P&L +1,032.15 (+103.2%)
Total Market Value: $14,315.45
Total Cost Basis: $4,500.00
Total Unrealized P&L: +9,815.45 (+218.1%)
Current Weights (by market value):
  NVDA: 14.3%
  AMD: 71.5%
  TSLA: 14.2%

--- Test 2: portfolio_id ไม่มีอยู่ ---
Error: ไม่พบ portfolio_id 'nonexistent_id' — portfolio ที่มีอยู่: demo, test_dead_ticker. ให้ถาม user ว่าหมายถึงอันไหน

--- Test 3: มี ticker ที่ fetch ราคาไม่ได้ ---
Portfolio: Test — delisted ticker (id: test_dead_ticker)
Positions:
  NVDA: 10 shares @ avg $150.00 → now $205.19 | MV $2,051.90 | P&L +551.90 (+36.8%)
Total Market Value: $2,051.90
Total Cost Basis: $1,500.00
Total Unrealized P&L: +551.90 (+36.8%

---
## Cell 7 — Agent setup (Groq + ReAct)

**ทำไม Groq ไม่ใช่ Gemini ตอน dev:** Gemini free tier ถอด quota ของ `gemini-2.0-flash` ออกแล้ว (`limit: 0` → 429 ตลอด) — dev ใน Colab ใช้ Groq แล้วค่อย swap เป็น Gemini ตอน deploy

**ทำไม `gpt-oss-120b` ไม่ใช่ Llama 3.3 70B:** reasoning model ที่ train มาเพื่อ agentic tasks → tool orchestration เสถียรกว่า + ถูกกว่า (`reasoning_effort="low"` พอสำหรับ tool routing และประหยัด output tokens)

⚠️ Trade-off: ภาษาไทยของ gpt-oss อ่อนกว่า Llama 3.3 (ที่รองรับไทย official) — ถ้า output ไทยเพี้ยน สลับ model ได้ด้วยการเปลี่ยน string เดียว

In [ ]:
# ============================================================
# Cell 7: Agent setup — ChatGroq + create_react_agent
# ============================================================

# Dev: Groq | Production: swap เป็น ChatGoogleGenerativeAI(model="gemini-2.0-flash")
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    reasoning_effort="low",   # พอสำหรับ tool orchestration, ประหยัด tokens
    api_key=userdata.get("GROQ_API_KEY"),
)


tools = [get_stock_price, get_stock_financials, get_hurst_exponent,
         analyze_portfolio_risk, track_portfolio, search_market_news]

SYSTEM_PROMPT = """You are a quantitative financial analyst assistant.
Always fetch real-time data before answering.
NEVER state any number that did not come from a tool result in this conversation.
If you lack data, call the appropriate tool or say you don't have it — do not estimate from memory.
For portfolio risk questions, use analyze_portfolio_risk.
Provide objective analysis with data. Note that this is not financial advice.
Do not give specific price targets, entry points, or stop-loss levels.
Respond in Thai mixed with English technical terms."""
agent_graph = create_react_agent(model, tools, prompt=SYSTEM_PROMPT)
print("✅ Agent graph ready")

✅ Agent graph ready


---
## Cell 8 — `run_financial_agent` (entry point + tracing wrapper)

หน้าที่ของ wrapper นี้:
1. **`@traceable`** — group ทุก sub-runs (LLM calls + tool calls) ไว้ใน 1 parent trace
2. **`callbacks=[tracer]`** — ส่ง LangGraph internal runs เข้า LangSmith แบบ explicit (ไม่พึ่ง env)
3. **`metadata` + `tags`** — filter ใน LangSmith UI ได้ตาม ticker / analysis_type
4. **`run_id` ใน return** — ดึงจาก `get_current_run_tree()` ภายใน function → ได้ ID ของ trace นี้เป๊ะ ๆ ไม่ต้อง query ย้อนหลัง (และคือ `trace_id` ที่ FastAPI endpoint ต้อง return ตาม spec)

In [ ]:
# ============================================================
# Cell 8: run_financial_agent — main entry point
# ============================================================

@traceable(
    name="financial_analyst_agent",
    run_type="chain",
    tags=["agent", "financial-analysis"],
    project_name=PROJECT_NAME,
    client=ls_client,
)
def run_financial_agent(
    query: str,
    tickers: list[str] = None,
    analysis_type: str = "general",
) -> dict:
    """Main entry point — group ทุก sub-runs ไว้ใน 1 parent trace"""
    config = RunnableConfig(
        run_name=f"query_{analysis_type}_{datetime.now().strftime('%H%M%S')}",
        callbacks=[tracer],                       # explicit tracer — ไม่พึ่ง env
        tags=[analysis_type] + (tickers or []),
        metadata={
            "query": query,
            "tickers": tickers or [],
            "analysis_type": analysis_type,
            "timestamp": datetime.now().isoformat(),
        },
    )

    inputs = {"messages": [HumanMessage(content=query)]}
    final_response = ""

    print(f"\n{'='*55}")
    print(f"🔍 Query: {query[:80]}...")
    print(f"{'='*55}")

    # stream_mode="values" → ได้ state เต็มทุก step, print ทุก message (Human/AI/Tool)
    for event in agent_graph.stream(inputs, config=config, stream_mode="values"):
        if "messages" in event:
            last = event["messages"][-1]
            last.pretty_print()
            if hasattr(last, "content") and last.content:
                final_response = last.content

    # ดึง run ID ของ trace นี้จากข้างใน — แม่นกว่า list_runs ย้อนหลัง
    rt = get_current_run_tree()
    return {
        "query": query,
        "response": final_response,
        "tickers": tickers,
        "analysis_type": analysis_type,
        "run_id": str(rt.id) if rt else None,
    }

print("✅ run_financial_agent ready")

✅ run_financial_agent ready


---
## Cell 9 — Test UC-1: วิเคราะห์หุ้นรายตัว

ลำดับการทำงาน:
1. **`tracing_context(enabled=True, client=ls_client)`** — เปิด tracing ให้ `@traceable` ทุกตัวในก้อนนี้ (จำเป็นเพราะเราไม่ได้ set env)
2. **`ls_client.flush()`** — บังคับส่ง pending traces ทันที (ปกติส่งแบบ background batch)
3. **Trace URLs** — private URL (เปิดดูเองใน workspace) + public URL จาก `share_run()` (แปะใน README ให้คนอื่นดูได้โดยไม่ต้อง login)

In [ ]:
# ============================================================
# Cell 9: Test UC-1 — single stock analysis
# ============================================================

with tracing_context(enabled=True, client=ls_client):
    result = run_financial_agent(
        query="วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)",
        tickers=["NVDA"],
        analysis_type="full_analysis",
    )

ls_client.flush()   # บังคับส่ง traces ก่อน query หา run

# ---------- Trace URLs ----------

time.sleep(5)       # เผื่อ server-side ingest

run_id = result["run_id"]
run = ls_client.read_run(run_id)
print(f"\n🔒 Private URL: {run.url}")

# Public URL — uncomment ถ้าต้องการ share (เช่นแปะใน README)
# shared_url = ls_client.share_run(run_id)
# print(f"🌐 Public URL: {shared_url}")


🔍 Query: วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)...
================================ Human Message =================================

วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (fc_5f46db24-4fd9-4d43-8185-609992736e4b)
 Call ID: fc_5f46db24-4fd9-4d43-8185-609992736e4b
  Args:
    ticker: NVDA
================================= Tool Message =================================
Name: get_stock_price

Ticker: NVDA
Price: $205.19 (Change: +0.16%)
52W Range: $142.03 – $236.54
P/E (TTM): 31.422665 | Forward P/E: 16.122227
Market Cap: $4969.9B
Position in 52W range: 67%
================================== Ai Message ==================================
Tool Calls:
  get_stock_financials (fc_d567b0c7-f118-49cc-a5f2-07af128c8d07)
 Call ID: fc_d567b0c7-f118-49cc-a5f2-07af128c8d07
  Args:
    ticker: NVDA
==============

---
## Cell 10–11 — Test UC-2a / UC-2b

tools จาก Cell 4–6:
- **UC-2a:** `run_financial_agent(query="ประเมิน risk ของ portfolio NVDA 50% AMD 30% TSLA 20%", analysis_type="portfolio_risk")`
- **UC-2b:** track P&L จาก `MOCK_PORTFOLIO`

## Cell 10: Test UC-2a — portfolio risk analysis

In [ ]:
# ============================================================
# Cell 10: Test UC-2a — portfolio risk analysis
# ============================================================
with tracing_context(enabled=True, client=ls_client):
    result = run_financial_agent(
        query="ประเมิน risk ของ portfolio NVDA 50% AMD 30% TSLA 20%",
        tickers=["NVDA", "AMD", "TSLA"],
        analysis_type="portfolio_risk",
    )
ls_client.flush()


🔍 Query: ประเมิน risk ของ portfolio NVDA 50% AMD 30% TSLA 20%...
================================ Human Message =================================

ประเมิน risk ของ portfolio NVDA 50% AMD 30% TSLA 20%
================================== Ai Message ==================================
Tool Calls:
  analyze_portfolio_risk (fc_82b50312-d05e-4237-bf43-d543ae69d73f)
 Call ID: fc_82b50312-d05e-4237-bf43-d543ae69d73f
  Args:
    portfolio: {"NVDA": 0.5, "AMD": 0.3, "TSLA": 0.2}
================================= Tool Message =================================
Name: analyze_portfolio_risk

Portfolio: {'NVDA': 0.5, 'AMD': 0.3, 'TSLA': 0.2}
Period: 1Y daily (250 trading days)
Annualized Return: +68.0%
Annualized Volatility: 36.7%
Sharpe Ratio: 1.73 (rf = 4.5%)
Sortino Ratio: 2.44
VaR 95% (daily): -3.84% | CVaR 95% (daily): -5.17%
Max Drawdown: -22.5%
Per-Ticker Volatility (annualized):
  AMD: 65.6%
  NVDA: 35.0%
  TSLA: 44.6%
Correlation Matrix:
       AMD  NVDA  TSLA
AMD   1.00  0.49  0.36
NVDA  0.4

---
## Utility — ดู runs ย้อนหลังในโปรเจกต์

`list_runs(is_root=True)` เหมาะกับ sanity check / ไล่ดู history หลาย runs — ต่างจาก `result["run_id"]` ที่ได้ run ของ query นั้นเป๊ะ ๆ

In [ ]:
# ============================================================
# Utility: list recent root runs
# ============================================================
runs = list(ls_client.list_runs(
    project_name=PROJECT_NAME,
    is_root=True,
    limit=5,
))
for r in runs:
    print(r.start_time, "|", r.name, "|", r.url)



2026-06-14 18:58:44.800875+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019ec780-25c0-7961-ae7f-4e8e514bd4d8?trace_id=019ec780-25c0-7961-ae7f-4e8e514bd4d8&start_time=2026-06-14T18:58:44.800875
2026-06-14 18:58:32.817391+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019ec77f-f6f1-7052-918e-45e932b0c02c?trace_id=019ec77f-f6f1-7052-918e-45e932b0c02c&start_time=2026-06-14T18:58:32.817391
2026-06-13 18:26:45.631818+00:00 | LangGraph | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019ec23c-80ff-7ed3-9c69-e7e6143bfdd4?trace_id=019ec23c-80ff-7ed3-9c69-e7e6143bfdd4&start_time=2026-06-13T18:26:45.631818
2026-06-13 18:26:35.591235+00:00 | LangGraph | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/pro

## Cell 11 — Test UC-2b

In [ ]:
# ============================================================
# Cell 11 — Test UC-2b
# ============================================================
with tracing_context(enabled=True, client=ls_client):
    result = run_financial_agent(
        query="พอร์ต demo ของฉันตอนนี้กำไรขาดทุนเท่าไหร่",
        tickers=["NVDA", "AMD", "TSLA"],
        analysis_type="portfolio_tracking",
    )
ls_client.flush()



## Test set — ครอบ over / under / co-trigger / regression

In [ ]:
# ============================================================
# Routing regression set — รันหลัง add search_market_news เข้า tools list
# เป้า: ยืนยัน news route ถูก + tool เดิม 1-5 ไม่ regress
# ============================================================
ROUTING_TESTS = [
    # ---- A) NEWS ควรถูกเรียก (under-trigger guard) ----
    {"query": "NVDA มีข่าวอะไรล่าสุดบ้าง",
     "expected": {"search_market_news"},
     "why": "ข่าวตรง ๆ — ถ้าไม่เรียก = under-trigger / ตอบจากความจำ (ผิด system prompt)"},

    {"query": "ทำไมหุ้น TSLA ร่วงช่วงนี้",
     "expected": {"search_market_news"},
     "why": "'ทำไม' = ต้องการ context เชิงเหตุการณ์ ไม่ใช่ตัวเลข"},

    {"query": "analyst มองหุ้น AMD ยังไงบ้างตอนนี้",
     "expected": {"search_market_news"},
     "why": "analyst commentary = qualitative → news"},

    # ---- B) NEWS ไม่ควรถูกเรียก (over-trigger guard) ----
    {"query": "ราคา NVDA ตอนนี้เท่าไหร่",
     "expected": {"get_stock_price"},
     "why": "ตัวเลขล้วน — news ไม่ควรโผล่ (news ช้าสุด เปลือง latency ฟรี)"},

    {"query": "P/E กับ profit margin ของ AMD",
     "expected": {"get_stock_financials"},
     "why": "fundamentals ล้วน → ห้าม news"},

    {"query": "ประเมิน risk พอร์ต NVDA 50% AMD 30% TSLA 20%",
     "expected": {"analyze_portfolio_risk"},
     "why": "risk metrics → ห้าม news, ห้ามแตกเป็น get_stock_price 3 ตัว"},

    {"query": "พอร์ต demo ตอนนี้กำไรขาดทุนเท่าไหร่",
     "expected": {"track_portfolio"},
     "why": "P&L พอร์ตที่มีอยู่ → ห้าม news, ห้าม analyze_portfolio_risk"},

    {"query": "ตอนนี้ NVDA เป็น trending หรือ mean-reverting",
     "expected": {"get_hurst_exponent"},
     "why": "regime = quant signal มีอยู่แล้ว → ห้ามไป search ข่าวแทน"},

    # ---- C) CO-TRIGGER: ต้องเรียกหลาย tool (จุดพังบ่อยสุด) ----
    {"query": "ราคา NVDA เท่าไหร่ และมีข่าวอะไรทำให้ขยับ",
     "expected": {"get_stock_price", "search_market_news"},
     "why": "ตัวเลข+ข่าว ต้องได้ทั้งคู่ — ขาดตัวใดตัวนึง = routing เพี้ยน"},

    {"query": "วิเคราะห์ NVDA แบบเต็ม: ราคา, fundamentals, regime, ข่าว",
     "expected": {"get_stock_price", "get_stock_financials",
                  "get_hurst_exponent", "search_market_news"},
     "why": "UC-1 ขยาย — เคย route 3 tools ถูก ต้องไม่ตกตัวไหนหลัง add news"},

    # ---- D) REGRESSION: UC เดิมเป๊ะ ๆ (ก่อนมี news ต้องเหมือนเดิม) ----
    {"query": "วิเคราะห์ NVDA ให้หน่อย: ราคา, fundamentals, Hurst",
     "expected": {"get_stock_price", "get_stock_financials", "get_hurst_exponent"},
     "why": "UC-1 ตัวเดิมเป๊ะ — news ไม่ควรแทรกเพราะไม่ได้ขอข่าว"},
]

In [ ]:
from langchain_core.messages import AIMessage

from groq import RateLimitError

def get_called_tools(query: str, max_retries: int = 3) -> set:
    """รัน agent จริง 1 query — retry เมื่อเจอ 429 (free tier TPM ตัน)"""
    for attempt in range(max_retries):
        try:
            called = set()
            inputs = {"messages": [HumanMessage(content=query)]}
            config = RunnableConfig(callbacks=[tracer])
            for event in agent_graph.stream(inputs, config=config, stream_mode="values"):
                for msg in event.get("messages", []):
                    if isinstance(msg, AIMessage) and msg.tool_calls:
                        for tc in msg.tool_calls:
                            called.add(tc["name"])
            return called
        except RateLimitError as e:
            wait = 8 * (attempt + 1)   # 8s, 16s, 24s — เผื่อ TPM window reset
            print(f"     ⏳ 429 — รอ {wait}s แล้ว retry (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
    raise RuntimeError(f"ยัง 429 หลัง retry {max_retries} ครั้ง — query: {query[:40]}")


def run_routing_tests():
    passed = failed = 0
    for i, t in enumerate(ROUTING_TESTS, 1):
        actual = get_called_tools(t["query"])
        ok = actual == t["expected"]

        # แยกแยะ failure mode ให้ชัด
        missing = t["expected"] - actual          # under-trigger / ตกหล่น
        extra   = actual - t["expected"]          # over-trigger / เรียกเกิน

        status = "✅" if ok else "❌"
        print(f"\n{status} [{i}] {t['query'][:55]}")
        print(f"     expected: {t['expected']}")
        print(f"     actual:   {actual}")
        if missing: print(f"     ⚠️ MISSING (under-trigger): {missing}")
        if extra:   print(f"     ⚠️ EXTRA (over-trigger):    {extra}")
        if not ok:  print(f"     why this matters: {t['why']}")

        passed += ok; failed += (not ok)
        time.sleep(2)   # กัน Groq rate limit

    print(f"\n{'='*55}\nRouting: {passed} passed, {failed} failed / {len(ROUTING_TESTS)}")


with tracing_context(enabled=True, client=ls_client):
    run_routing_tests()
ls_client.flush()


✅ [1] NVDA มีข่าวอะไรล่าสุดบ้าง
     expected: {'search_market_news'}
     actual:   {'search_market_news'}

✅ [2] ทำไมหุ้น TSLA ร่วงช่วงนี้
     expected: {'search_market_news'}
     actual:   {'search_market_news'}

✅ [3] analyst มองหุ้น AMD ยังไงบ้างตอนนี้
     expected: {'search_market_news'}
     actual:   {'search_market_news'}

✅ [4] ราคา NVDA ตอนนี้เท่าไหร่
     expected: {'get_stock_price'}
     actual:   {'get_stock_price'}

❌ [5] P/E กับ profit margin ของ AMD
     expected: {'get_stock_financials'}
     actual:   {'get_stock_price', 'get_stock_financials'}
     ⚠️ EXTRA (over-trigger):    {'get_stock_price'}
     why this matters: fundamentals ล้วน → ห้าม news

✅ [6] ประเมิน risk พอร์ต NVDA 50% AMD 30% TSLA 20%
     expected: {'analyze_portfolio_risk'}
     actual:   {'analyze_portfolio_risk'}

✅ [7] พอร์ต demo ตอนนี้กำไรขาดทุนเท่าไหร่
     expected: {'track_portfolio'}
     actual:   {'track_portfolio'}

✅ [8] ตอนนี้ NVDA เป็น trending หรือ mean-reverting
     expected: {'

### Note

จุดที่อยากเน้น:

over-trigger ที่เรากลัวตอน add tool ที่ 6 (news ลาม) ไม่เกิดเลย — case 1-3 news route แม่น, case 4-8 single-tool ไม่มี news แทรก, case 9-10 news เข้าเฉพาะตอนขอข่าว เป้าหมายหลักของ regression รอบนี้ (พิสูจน์ว่า add news ไม่พังของเดิม) ผ่านสมบูรณ์ case 5 เป็นปัญหา price/financials ที่มีมาก่อน news ด้วยซ้ำ ไม่เกี่ยวกับงานรอบนี้

In [ ]:
print("เช็ค consistency case 5 (P/E + margin):")
for i in range(5):
    print(f"  รอบ {i+1}: {get_called_tools('P/E กับ profit margin ของ AMD')}")
    time.sleep(5)

เช็ค consistency case 5 (P/E + margin):
  รอบ 1: {'get_stock_price', 'get_stock_financials'}
  รอบ 2: {'get_stock_price', 'get_stock_financials'}
  รอบ 3: {'get_stock_price', 'get_stock_financials'}
  รอบ 4: {'get_stock_price', 'get_stock_financials'}
  รอบ 5: {'get_stock_price', 'get_stock_financials'}


### Note
known limitation — แล้วมันกลายเป็นข้อดีตอนสัมภาษณ์ด้วยซ้ำ ถ้าผู้สัมภาษณ์เปิด trace เจอ แล้วคุณอธิบายได้ว่า:

"P/E + margin เรียก get_stock_price เกินมา 5/5 — รู้ตัว วินิจฉัยแล้วว่าเป็นเพราะ P/E ผูกกับราคาเชิงความหมาย docstring-level negative routing งัดไม่ขึ้น ตัดสินใจไม่ดันต่อเพราะ over-fetch ตัวนี้ benign (price เป็น tool เร็ว/ถูกสุด) และของแรงกว่ามีความเสี่ยง regress multi-tool UC ที่เพิ่งแก้สมดุล — ถ้าจำเป็นจริงทางแก้คือ conditional routing ใน StateGraph (v2) ไม่ใช่ docstring"

# SQLite

In [3]:
# ============================================================
# Cell A: Install SQLAlchemy + aiosqlite + nest_asyncio
# ============================================================
!pip install -q sqlalchemy aiosqlite nest_asyncio

import asyncio
import nest_asyncio
nest_asyncio.apply()   # ให้ asyncio.run() ทำงานใน Colab event loop ได้

import sqlalchemy
print("✅ SQLAlchemy:", sqlalchemy.__version__)
print("✅ nest_asyncio applied")

✅ SQLAlchemy: 2.0.50
✅ nest_asyncio applied


In [4]:
# ============================================================
# Cell B: db.py — SQLAlchemy models + async engine + session
# ============================================================
import uuid
from datetime import datetime

from sqlalchemy import String, Float, ForeignKey, Text, func
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession, async_sessionmaker

DB_PATH = "/content/portfolio.db"
DATABASE_URL = f"sqlite+aiosqlite:///{DB_PATH}"

engine = create_async_engine(DATABASE_URL, echo=False)
AsyncSessionLocal = async_sessionmaker(engine, expire_on_commit=False)


class Base(DeclarativeBase):
    pass


class Portfolio(Base):
    __tablename__ = "portfolios"

    portfolio_id: Mapped[str] = mapped_column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    name:         Mapped[str] = mapped_column(Text, nullable=False)
    created_at:   Mapped[str] = mapped_column(Text, default=lambda: datetime.utcnow().isoformat())

    positions: Mapped[list["Position"]] = relationship(
        "Position", back_populates="portfolio", cascade="all, delete-orphan"
    )


class Position(Base):
    __tablename__ = "positions"

    position_id:  Mapped[str] = mapped_column(String, primary_key=True, default=lambda: str(uuid.uuid4()))
    portfolio_id: Mapped[str] = mapped_column(String, ForeignKey("portfolios.portfolio_id", ondelete="CASCADE"))
    ticker:       Mapped[str] = mapped_column(String, nullable=False)
    shares:       Mapped[float] = mapped_column(Float, nullable=False)
    avg_cost:     Mapped[float] = mapped_column(Float, nullable=False)
    created_at:   Mapped[str] = mapped_column(Text, default=lambda: datetime.utcnow().isoformat())
    updated_at:   Mapped[str] = mapped_column(Text, default=lambda: datetime.utcnow().isoformat())

    portfolio: Mapped["Portfolio"] = relationship("Portfolio", back_populates="positions")


async def init_db():
    """สร้าง tables ทั้งหมด (idempotent — รันซ้ำได้)"""
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)


# รัน init ทันที
import asyncio
asyncio.run(init_db())
print("✅ DB initialized:", DB_PATH)

✅ DB initialized: /content/portfolio.db


In [5]:
# ============================================================
# Cell C: Seed — INSERT MOCK_PORTFOLIOS เข้า DB
# (idempotent: เช็ค portfolio_id ก่อน insert เพื่อกัน duplicate)
# ============================================================
from sqlalchemy import select

SEED_DATA = {
    "demo": {
        "name": "Demo Tech Portfolio",
        "positions": [
            {"ticker": "NVDA", "shares": 10, "avg_cost": 150.0},
            {"ticker": "AMD",  "shares": 20, "avg_cost": 100.0},
            {"ticker": "TSLA", "shares":  5, "avg_cost": 200.0},
        ],
    },
    "test_dead_ticker": {
        "name": "Test — delisted ticker",
        "positions": [
            {"ticker": "NVDA",       "shares": 10, "avg_cost": 150.0},
            {"ticker": "ZZZFAKE123", "shares":  5, "avg_cost":  50.0},
        ],
    },
}


async def seed_db():
    async with AsyncSessionLocal() as session:
        for pf_id, pf_data in SEED_DATA.items():
            # เช็คว่ามีอยู่แล้วไหม — กัน duplicate เมื่อรัน cell ซ้ำ
            existing = await session.get(Portfolio, pf_id)
            if existing:
                print(f"  ⏭️  '{pf_id}' already exists — skip")
                continue

            pf = Portfolio(
                portfolio_id=pf_id,
                name=pf_data["name"],
            )
            session.add(pf)

            for p in pf_data["positions"]:
                session.add(Position(
                    portfolio_id=pf_id,
                    ticker=p["ticker"],
                    shares=p["shares"],
                    avg_cost=p["avg_cost"],
                ))

        await session.commit()
        print("✅ Seed complete")


asyncio.run(seed_db())

✅ Seed complete


/tmp/ipykernel_430/2815511122.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at:   Mapped[str] = mapped_column(Text, default=lambda: datetime.utcnow().isoformat())
/tmp/ipykernel_430/2815511122.py:42: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at:   Mapped[str] = mapped_column(Text, default=lambda: datetime.utcnow().isoformat())
/tmp/ipykernel_430/2815511122.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  updated_at:   Mapped[str] = mapped_column(Text, default=lambda: datetime.utcnow().is

In [6]:
# ============================================================
# Cell D: swap _load_positions() — dict → SQLite query
# แทนที่ฟังก์ชันเดิมใน Cell 6 (interface เดิมไม่เปลี่ยน)
# ============================================================
from sqlalchemy import select

async def _load_positions_async(portfolio_id: str):
    """Query DB — คืน (name, positions_list) เหมือน interface เดิมทุกอย่าง"""
    async with AsyncSessionLocal() as session:
        pf = await session.get(Portfolio, portfolio_id)
        if pf is None:
            raise KeyError(portfolio_id)

        result = await session.execute(
            select(Position).where(Position.portfolio_id == portfolio_id)
        )
        rows = result.scalars().all()

        positions = [
            {"ticker": r.ticker, "shares": r.shares, "avg_cost": r.avg_cost}
            for r in rows
        ]
        return pf.name, positions


def _load_positions(portfolio_id: str):
    """Sync wrapper — ให้ _track_portfolio_logic เรียกได้เหมือนเดิม
    (ไม่ต้องแก้ tool หรือ agent graph)"""
    return asyncio.run(_load_positions_async(portfolio_id))


print("✅ _load_positions() swapped → SQLite")

✅ _load_positions() swapped → SQLite
